# 🚀 Fine-Tuning GPT-2 Small with LoRA & GPU Acceleration

Welcome to the **GPT-2 Production-Level Fine-Tuning** workspace! This notebook guides you through running the PyTorch training pipeline from scratch on a **Google Colab T4 GPU**.

### 🌟 Features
- **Supervised Fine-Tuning (SFT)** with Alpaca-style instruction dataset parsing.
- **Parameter-Efficient LoRA (Low-Rank Adaptation)** wrapping query and value projections.
- **Hardware Optimizations**: FP16 Automatic Mixed Precision (AMP) and Gradient Accumulation.
- **Telemetry**: Real-time training loss logging.

## 1. ⚙️ Setup & Environment Configuration

First, we check that we have a GPU active, clone the repository, and install the required packages.

In [ ]:
# Verify GPU accessibility
!nvidia-smi

In [ ]:
# Clone the repository (replace with your fork or directory if needed)
# !git clone https://github.com/amoghsamadhiya779-afk/GPT-PRODUCTION-LEVEL.git
# %cd GPT-PRODUCTION-LEVEL

In [ ]:
# Install dependencies
!pip install -r requirements.txt

## 2. 📂 Prepare the Dataset

You can upload your own custom data to Colab or use a default mock dataset. 
Let's write a sample `custom_instructions.json` file to demonstrate how the instruction parser handles Alpaca-style instruction datasets.

In [ ]:
import json

# Create a custom instruction dataset
custom_data = [
    {
        "instruction": "Who is Amogh?",
        "input": "",
        "output": "Amogh is a Backend & MLOps engineer specializing in distributed systems, PyTorch deep learning, and production scale architectures. He is the founder of Omkala Publications."
    },
    {
        "instruction": "What are Omkala Publications?",
        "input": "",
        "output": "Omkala Publications is a publishing entity founded by Amogh, specializing in distributing literature, technical materials, and books."
    },
    {
        "instruction": "Explain the self-attention mechanism.",
        "input": "",
        "output": "Self-attention is a core mechanism in the Transformer architecture where input tokens are mapped into Query, Key, and Value vectors. Attention weights are computed using the scaled dot product of Queries and Keys, allowing tokens to weight context from all other tokens in parallel."
    }
]

with open("data/custom_instructions.json", "w", encoding="utf-8") as f:
    json.dump(custom_data, f, indent=2)

print("Saved custom instruction dataset to data/custom_instructions.json")

## 3. 🚀 Run the LoRA Finetuning Pipeline

We will now launch the training loop. We configure:
- `--lora`: Only train adapter parameters (extremely memory efficient).
- `--data_type instruction`: Enable target loss masking (masking prompt tokens with `-100`).
- `--use_amp`: Enable FP16 Mixed Precision for high-speed computation.
- `--accum_steps 2`: Accumulate gradients over 2 micro-batches before optimization step.

In [ ]:
# Run finetuning (using PyTorch AMP + LoRA)
!python training/train.py \
    --config configs/gpt2_small.yaml \
    --data data/custom_instructions.json \
    --data_type instruction \
    --lora \
    --lora_r 4 \
    --lora_alpha 8.0 \
    --accum_steps 2 \
    --use_amp

## 4. 🧪 Verify Checkpoint & Inference Serving

Once training completes, the LoRA checkpoint is saved to `checkpoints/best_model.pt`.
We can instantiate the `GPTInferenceEngine` to verify that the model correctly loads the LoRA adapter and outputs the custom facts we trained it on.

In [ ]:
from app.inference import GPTInferenceEngine

# Load the newly trained LoRA checkpoint
engine = GPTInferenceEngine("checkpoints/best_model.pt")

# Generate text
res = engine.generate("Who is Amogh?", max_new_tokens=40, temperature=0.7, top_p=0.9, repetition_penalty=1.2)
print("\n--- Generated Text ---")
print(res["generated_text"])
print(f"Latency: {res['time_taken_seconds']:.3f}s | Speed: {res['tokens_per_second']:.1f} t/s")

## 5. 📥 Deploying Your Checkpoint

To use this adapter in your local app or Hugging Face Space:
1. Download `checkpoints/best_model.pt` from the Colab file browser.
2. Copy the file into the `checkpoints/` folder of your project repository.
3. Re-launch the server! The server will automatically detect the checkpoint, inject the LoRA layers on the base GPT-2 model, and serve the adapter model.